# Model 3 = Deep nueral network

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)


In [2]:
customer_data = pd.read_excel('../Datasets/cleaned_customers_churn_data.xlsx')
customer_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Germany,Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,1


In [3]:
x = customer_data.drop(columns=['Exited'])
y = customer_data['Exited']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((8000, 11), (2000, 11), (8000,), (2000,))

In [5]:
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [8]:

from tensorflow.keras.backend import clear_session
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

architectures = [
    (22, 11),
    (32, 16),
    (64, 32),
    (16, 8)
]

results = []

for h1, h2 in architectures:

    clear_session()

    model = Sequential()

    # Hidden Layer 1
    model.add(Dense(units=h1, activation='relu', input_dim=11))

    # Hidden Layer 2
    model.add(Dense(units=h2, activation='relu'))

    # Output Layer
    model.add(Dense(units=1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        x_train,
        y_train,
        epochs=100,
        batch_size=32,
        verbose=0
    )

    y_pred = model.predict(x_test, verbose=0)
    y_pred = (y_pred > 0.5)

    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)

    results.append([
        h1,
        h2,
        round(acc, 4),
        round(recall, 4),
        round(f1, 4),
        round(auc, 4)
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        'Hidden1',
        'Hidden2',
        'Accuracy',
        'Recall',
        'F1 Score',
        'ROC-AUC'
    ]
)

print(results_df)

c:\softwers\anaconda\installed_here\envs\churn_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\softwers\anaconda\installed_here\envs\churn_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\softwers\anaconda\installed_here\envs\churn_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the mode

   Hidden1  Hidden2  Accuracy  Recall  F1 Score  ROC-AUC
0       22       11    0.8630  0.5089    0.5935   0.7293
1       32       16    0.8635  0.4733    0.5767   0.7161
2       64       32    0.8555  0.4809    0.5667   0.7140
3       16        8    0.8655  0.4936    0.5906   0.7250


Multiple Deep Neural Network architectures were evaluated using different hidden layer configurations. The architecture with 22 neurons in the first hidden layer and 11 neurons in the second hidden layer achieved the best overall performance, with the highest Recall (50.89%), F1 Score (59.35%), and ROC-AUC (72.93%). Since the business objective is to identify customers likely to churn, Recall and F1 Score were prioritized over Accuracy. Increasing the number of neurons beyond this point did not improve performance and instead increased model complexity.